In [4]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [10]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active device: {device}")

Active device: mps


In [2]:
df = pd.read_csv('fmnist_small.csv')
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

In [4]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [5]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [6]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [7]:
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle = False, pin_memory=True)

In [8]:
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )

  def forward(self, x):
    return self.model(x)


In [9]:
learning_rate = 0.1
epochs = 100

In [10]:
model = MyNN(X_train.shape[1])
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= learning_rate)

In [11]:
for epoch in range(epochs):
  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)

    loss = criterion(outputs, batch_labels)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()
    total_epoch_loss += loss.item()

  average_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch+1}, Loss: {average_loss}')




Epoch: 1, Loss: 1.3376668202877045
Epoch: 2, Loss: 0.7769730232159296
Epoch: 3, Loss: 0.6486381661891937
Epoch: 4, Loss: 0.5904474651813507
Epoch: 5, Loss: 0.5330705397327741
Epoch: 6, Loss: 0.5070943200588226
Epoch: 7, Loss: 0.47798695663611096
Epoch: 8, Loss: 0.4563480198383331
Epoch: 9, Loss: 0.4266120778520902
Epoch: 10, Loss: 0.4105663736661275
Epoch: 11, Loss: 0.38902041733264925
Epoch: 12, Loss: 0.37695423180858295
Epoch: 13, Loss: 0.3603440902630488
Epoch: 14, Loss: 0.3419398063917955
Epoch: 15, Loss: 0.33983191673954327
Epoch: 16, Loss: 0.3159318087995052
Epoch: 17, Loss: 0.31098644226789474
Epoch: 18, Loss: 0.30416499838232997
Epoch: 19, Loss: 0.2939030587176482
Epoch: 20, Loss: 0.2734547992547353
Epoch: 21, Loss: 0.2691309749335051
Epoch: 22, Loss: 0.2655207734306653
Epoch: 23, Loss: 0.25327663533389566
Epoch: 24, Loss: 0.2520088792592287
Epoch: 25, Loss: 0.24160942112406095
Epoch: 26, Loss: 0.23981781082848708
Epoch: 27, Loss: 0.21824247412383557
Epoch: 28, Loss: 0.21756500

In [12]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [13]:
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      output = model(batch_features)
      _, predicted = torch.max(output, 1)
      total += batch_labels.shape[0]

      correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {correct/total * 100:.2f}%')

Accuracy: 84.50%


In [14]:
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    output = model(batch_features)
    _, predicted = torch.max(output, 1)
    total += batch_labels.shape[0]

    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {correct/total * 100:.2f}%')

Accuracy: 99.06%
